# Stress tests
CIFAR-10 stress tests for the magazine article, each changing one setting of the main runs in setup_remote.ipynb; results go to a separate Results/stress folder.


Prepare the project and its imports, as in setup_remote.ipynb.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

ON_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
REPOSITORY_URL = "https://github.com/s4meer-git/BPDC_FL_Framework.git"

if ON_COLAB:
    PROJECT_ROOT = Path("/content/FLFrameWork")
    if (PROJECT_ROOT / ".git").exists():
        if "configuration" in sys.modules or "main" in sys.modules:
            raise RuntimeError("Restart the Python session before fetching code so old functions are not kept in memory.")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "JPNotebooks" / "requirements.txt")], check=True)
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "JPNotebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "configuration.py").is_file():
    raise FileNotFoundError("Open this notebook from the project folder, or set PROJECT_ROOT to your checkout.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from configuration import configuration
from support_modules.experiment import prepare_runs, run_experiments, comparison_csvs

print("Project:", PROJECT_ROOT)
print("Device:", configuration().device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Stress-test results are saved as Results/stress/<test>/cifar10/RUN1, RUN2, RUN3/<selfish|non_selfish>/<method>.csv.


In [ ]:
RESULTS_ROOT = PROJECT_ROOT / "Results" / "stress"
BACKUP_ROOT = None

if ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    BACKUP_ROOT = Path("/content/drive/MyDrive/FLFrameWork/Results")

print("Working results:", RESULTS_ROOT)
print("Backup:", BACKUP_ROOT if BACKUP_ROOT is not None else "Local run; CSVs stay on this computer.")

Choose the tests and seeds; ONLY_SEEDS limits this session to some seeds while keeping RUN numbering, so several sessions can share the work.


In [ ]:
import copy

config = configuration()

# "cifar10" for the paper's stress tests; "tinyimagenet" repeats them on TinyImageNet.
DATASET = "cifar10"
METHODS = ["fedavg", "fcfl", "fedcda", "fedmut", "fairrfl"]
SEEDS = [42, 43, 44]
ONLY_SEEDS = None
RERUN_COMPLETED = False


# True skips the selfish scenario of every test and runs only its honest (non_selfish) runs.
HONEST_ONLY = False


# The same common settings as the main CIFAR-10 runs in setup_remote.ipynb; each stress test changes one setting.
config.num_of_clients = 30
config.rounds = 40
config.local_epochs = 5
config.batch_size = 64
config.lr = 0.01
config.dirichlet_alpha = 0.5
MOMENTUM = {"mnist": 0.5, "cifar10": 0.9, "tinyimagenet": 0.9}
config.selfish_fraction = 0.05
config.selfish_strength = 0.7
config.selfish_start_round = 3
config.fairrfl_q = 1.0
config.fairrfl_tau = 2.5
config.fairrfl_dynamic_q = True
config.fairrfl_lipschitz = None
config.fedcda_warmup = 10
config.fcfl_alpha = 0.5
config.fcfl_fraction = 1.0


# Each test: its results folder name, the methods it runs, the scenarios (selfishness on or off) and the one changed setting.
TESTS = {
    "quiet_attacker":         {"methods": METHODS,   "selfish": [True],        "settings": {"selfish_strength": 0.05}},
    "fairrfl_screen_only":    {"methods": ["fairrfl"], "selfish": [False, True], "settings": {"fairrfl_q": 0.0}},
    "fairrfl_weighting_only": {"methods": ["fairrfl"], "selfish": [False, True], "settings": {"fairrfl_tau": 1e9}},
    "fairrfl_screen_before_weighting": {"methods": ["fairrfl"], "selfish": [False, True], "settings": {"fairrfl_screen_before_weighting": True}},
    "fairrfl_screen_sqrt_steps": {"methods": ["fairrfl"], "selfish": [False, True], "settings": {"fairrfl_screen_before_weighting": True, "fairrfl_screen_sqrt_steps": True}},
    "fedavg_norm_clip":       {"methods": ["fedavg"],  "selfish": [False, True], "settings": {"fedavg_norm_clip": True}},
    "nine_attackers":         {"methods": METHODS,   "selfish": [True],        "settings": {"selfish_fraction": 0.3}},
    "skew_alpha_0.1":         {"methods": METHODS,   "selfish": [False, True], "settings": {"dirichlet_alpha": 0.1}},
}
SELECTED_TESTS = list(TESTS)

runs = []
for name in SELECTED_TESTS:
    test = TESTS[name]
    for selfish in test["selfish"]:
        if HONEST_ONLY and selfish:
            continue
        test_config = copy.deepcopy(config)
        test_config.selfishness = selfish
        for setting, value in test["settings"].items():
            setattr(test_config, setting, value)
        planned = prepare_runs(test_config, [DATASET], test["methods"], RESULTS_ROOT / name, MOMENTUM, SEEDS)
        runs += [run for run in planned if ONLY_SEEDS is None or run[0].seed in ONLY_SEEDS]


# Every run writes one CSV here: Results/stress/<test>/cifar10/RUN<n>/<selfish|non_selfish>/<method>.csv; RUN1 is the first seed in SEEDS.
print("Results folder:", RESULTS_ROOT)
for run_config, folder in runs:
    print(" ", folder.relative_to(RESULTS_ROOT) / f"{run_config.method}.csv", " (seed", str(run_config.seed) + ")")
print("Runs:", len(runs))


Run the selected tests; completed runs are skipped, diverged runs are recorded, and incomplete runs restart from round 1.


In [ ]:
if ON_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU runtime before training; loading CSVs and summaries do not need one.")

run_summary = run_experiments(runs, backup_root=BACKUP_ROOT, rerun_completed=RERUN_COMPLETED)
display(run_summary)


Run this after every seed has finished to save mean ± sample standard deviation for each test and scenario.


In [ ]:
from analysis.results import seed_summary
from analysis.plots import format_seed_summary
from support_modules.experiment import backup_csv, backup_path


# One summary per test and scenario, saved as summary_<scenario>.csv in that test's dataset folder.
comparisons = {}
for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    comparisons.setdefault((folder.parent.parent, folder.name), {})[folder.parent.name] = csv_data

for (dataset_folder, scenario), csv_by_seed in comparisons.items():
    means, deviations, per_seed, partial = seed_summary(csv_by_seed)
    summary = format_seed_summary(means, deviations)
    summary_path = dataset_folder / f"summary_{scenario}.csv"
    summary.to_csv(summary_path)
    per_seed.to_csv(dataset_folder / f"summary_{scenario}_per_seed.csv", index=False)
    print(dataset_folder.relative_to(RESULTS_ROOT), scenario, "| RUNs:", sorted(csv_by_seed))
    for method, missing in partial.items():
        print(f"{method} diverged for seeds {missing}; it is left out of the mean ± std table.")
    display(summary)
    backup_csv(summary_path, backup_path(dataset_folder, BACKUP_ROOT))
